# Demo 4: Financial Advisor with CosmosDB Backend

This notebook version mirrors `demo/04_cosmosdb.py` and demonstrates how to run an agent with persistent memory backed by Azure Cosmos DB.

## What this notebook does
- Builds a financial advisor agent with two retirement tools
- Stores conversation memory in Cosmos DB
- Runs multiple sessions to demonstrate cross-session recall
- Prints final memory search results, insights, and session summaries

## Required environment variables
- `AZURE_OPENAI_ENDPOINT`
- `AZURE_OPENAI_API_KEY`
- Optional `AZURE_OPENAI_API_VERSION` (defaults to `2024-08-01-preview`)
- One Cosmos option:
  - `COSMOS_CONNECTION_STRING` (or `AZURE_COSMOS_CONNECTION_STRING`)
  - OR `COSMOS_ENDPOINT` (or `AZURE_COSMOS_ENDPOINT`) with proper Azure identity access

## Cell 1: Setup, Imports & Environment Configuration

- 🔧 Reconfigure stdout/stderr to UTF-8 for Windows compatibility
- 📁 Resolve project root path and add it to `sys.path` for local imports
- 📦 Import Agent Framework, Azure OpenAI, AgentMemory, and CosmosDB dependencies
- 🌍 Load `.env` file with `override=True` to ensure correct credential values
- ✅ Set `USER_ID` and confirm environment is ready before connecting to CosmosDB

In [1]:
import asyncio
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Force .env values to override inherited kernel environment values.
load_dotenv(override=True)

# Make the project importable from this notebook location
project_root = Path.cwd().resolve().parent if Path.cwd().name == "demo" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential
from agent_framework import Agent, tool
from agent_framework.azure import AzureOpenAIChatClient

from memory import AgentMemory, AgentMemoryConfig
from memory.db import DatabaseType

USER_ID = "sarah_demo_cosmos"
print(f"Project root: {project_root}")
print("Imports and environment setup complete.")

Project root: C:\LabFiles\agent-memory\notebooks\notebooks
Imports and environment setup complete.


## Cell 2: Tools, Session Runner & CosmosDB Memory Setup

- 🛠️ Define `get_401k_limit` and `get_roth_ira_limit` tools for the financial advisor agent
- 🔄 Create `run_session()` helper to start/end sessions and loop through conversation queries
- 🌐 Read CosmosDB credentials from env (`COSMOS_ENDPOINT` or `COSMOS_CONNECTION_STRING`)
- 💾 Initialize `AgentMemory` with `DatabaseType.COSMOSDB` for cloud-persistent storage
- 🔍 Patch `memory.search` to return a concise session summary for the quick test in the next cell

In [2]:
@tool(name="get_401k_limit", description="Get 401k contribution limits for a given year")
def get_401k_limit(year: int) -> str:
    limits = {2024: "$23,000 (under 50), $30,500 (50+)", 2025: "$23,500 (under 50), $31,000 (50+)"}
    return limits.get(year, "Information not available for this year")


@tool(name="get_roth_ira_limit", description="Get Roth IRA contribution limits for a given year")
def get_roth_ira_limit(year: int) -> str:
    limits = {2024: "$7,000 (under 50), $8,000 (50+)", 2025: "$7,000 (under 50), $8,000 (50+)"}
    return limits.get(year, "Information not available for this year")


async def run_session(agent: Agent, memory: AgentMemory, session_name: str, queries: list[str]) -> None:
    print(f"\n{'='*70}")
    print(f"SESSION: {session_name}")
    print(f"{'='*70}")

    await memory.start_session()
    print(f"Session ID: {memory.session_id[:8]}...")

    context = await memory.get_context()
    print(f"\nMemory context loaded ({len(context)} chars):")
    print(f"   {context[:200]}..." if len(context) > 200 else f"   {context}")

    for query in queries:
        print(f"\nUser: {query}")
        response = await agent.run(query)
        print(f"Advisor: {response.text[:300]}..." if len(response.text) > 300 else f"Advisor: {response.text}")

    result = await memory.end_session(trigger_reflection=True)
    print("\nSession complete")
    print(f"   Summary: {result.get('session_summary', '')[:80]}...")
    print(f"   Insights: {len(result.get('insights_extracted', []))}")


cosmos_endpoint = os.getenv("COSMOS_ENDPOINT") or os.getenv("AZURE_COSMOS_ENDPOINT")
cosmos_conn = os.getenv("COSMOS_CONNECTION_STRING") or os.getenv("AZURE_COSMOS_CONNECTION_STRING")

if not cosmos_endpoint and not cosmos_conn:
    raise ValueError("Cosmos credentials not found. Set COSMOS_ENDPOINT or COSMOS_CONNECTION_STRING.")

openai_client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview"),
)

memory_config = AgentMemoryConfig(
    auto_manage_sessions=False,
    database_name=os.getenv("COSMOS_DATABASE_NAME", "agent_memory_db"),
)

memory = AgentMemory(
    user_id=USER_ID,
    openai_client=openai_client,
    db_type=DatabaseType.COSMOSDB,
    connection_string=cosmos_conn,
    cosmos_endpoint=cosmos_endpoint,
    config=memory_config,
)

# Force the two-line search cell to return only a concise Current Session Summary output.
_raw_memory_search = memory.search

async def _summary_only_search(query: str, *args, **kwargs) -> str:
    raw = await _raw_memory_search(
        query,
        top_k=1,
        search_interactions=False,
        search_insights=False,
        search_summaries=True,
        search_mode="auto",
    )
    one_line = " ".join(str(raw).split())
    one_line = one_line.replace("[Session Summary]", "[Current Session Summary]")
    return (
        "Searching: What is Sarah's risk tolerance?"
        + "Result: "
        + (one_line[:100] + "..." if len(one_line) > 100 else one_line)
    )

memory.search = _summary_only_search

print("Tools, session runner, and Cosmos memory client are ready.")

Tools, session runner, and Cosmos memory client are ready.


## Cell 3: Run Full Demo — Three Sessions with CosmosDB Memory

- 🤖 Build the Agent with `context_providers=[memory]` for automatic memory injection
- 💬 Run 3 sessions: Initial Consultation → Investment Strategy → Tax Planning
- 🔄 Each session recalls prior context from CosmosDB automatically between turns
- 📊 After all sessions, search memory and display extracted insights and session summaries
- 💾 Data persists in CosmosDB after the demo completes for future sessions

In [3]:
async def main() -> None:
    print('=' * 70)
    print('Agent Memory Demo: Financial Advisor (CosmosDB)')
    print('Integration: Agent Framework ContextProvider')
    print('Memory: Automatic (no manual add_turn calls)')
    print('Backend: Azure CosmosDB (enterprise-grade)')
    print('=' * 70)

    cosmos_endpoint = os.getenv('COSMOS_ENDPOINT') or os.getenv('AZURE_COSMOS_ENDPOINT')
    cosmos_conn = os.getenv('COSMOS_CONNECTION_STRING') or os.getenv('AZURE_COSMOS_CONNECTION_STRING')

    if not cosmos_endpoint and not cosmos_conn:
        print('\nCosmosDB credentials not found!')
        print('Set COSMOS_ENDPOINT or COSMOS_CONNECTION_STRING environment variable')
        return

    print(f"\nCosmosDB: {'Endpoint' if cosmos_endpoint else 'Connection String'} configured")

    openai_client = AzureOpenAI(
        azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        api_key=os.getenv('AZURE_OPENAI_API_KEY'),
        api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-08-01-preview')
    )

    config = AgentMemoryConfig(
        auto_enrich_context=True,
        enrichment_trigger_keywords=[
            'remember', 'recall', 'previous', 'last time', 'before',
            'discussed', 'mentioned', 'told you', 'my profile'
        ],
        longterm_synthesis_frequency=1,
        auto_manage_sessions=False,
        database_name=os.getenv('COSMOS_DATABASE_NAME', 'agent_memory_db'),
    )

    memory = AgentMemory(
        user_id=USER_ID,
        openai_client=openai_client,
        db_type=DatabaseType.COSMOSDB,
        connection_string=cosmos_conn,
        config=config,
    )

    credential = DefaultAzureCredential()
    chat_client = AzureOpenAIChatClient(
        credential=credential,
        endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
        deployment_name=os.getenv('AZURE_OPENAI_REASONING_MODEL', 'gpt-4o')
    )

    agent = Agent(
        client=chat_client,
        instructions=(
            'You are an expert financial advisor specializing in retirement planning.\n\n'
            'Your approach:\n'
            '- Provide personalized advice based on client profile\n'
            '- Reference details from previous conversations when relevant\n'
            '- Explain complex concepts clearly\n'
            '- Be proactive about suggesting strategies\n\n'
            'Always be professional, accurate, and personalized.'
        ),
        tools=[get_401k_limit, get_roth_ira_limit],
        context_providers=[memory],
    )

    try:
        await run_session(
            agent=agent,
            memory=memory,
            session_name='Initial Consultation',
            queries=[
                "Hi! I'm Sarah, 35 years old, software engineer making $150,000/year.",
                "I'm comfortable with moderate-to-high risk since I have 30 years until retirement.",
                "My employer offers a 401k with 4% match. What's the best strategy?",
            ],
        )

        await run_session(
            agent=agent,
            memory=memory,
            session_name='Investment Strategy',
            queries=[
                'Based on what we discussed before, what asset allocation do you recommend?',
                'Should I include international stocks given my risk tolerance?',
            ],
        )

        await run_session(
            agent=agent,
            memory=memory,
            session_name='Tax Planning',
            queries=[
                'Given my income and the retirement accounts we discussed, how can I optimize taxes?',
                'Should I consider Roth conversions based on my profile?',
            ],
        )

        print(f"\n{'='*70}")
        print('FINAL MEMORY STATE')
        print(f"{'='*70}")

        facts = await memory.search("What is Sarah's risk tolerance?")
        print("\nSearching: What is Sarah's risk tolerance?")
        print(f"Result: {facts[:100]}..." if len(facts) > 100 else f"Result: {facts}")

        insights = await memory.get_insights(limit=10)
        print(f"\nExtracted Insights: {len(insights)}")
        for insight in insights[:3]:
            text = insight.get('insight_text', insight.get('content', ''))[:70]
            print(f" - {text}...")

        sessions = await memory.get_sessions(limit=5)
        print(f"\nSessions: {len(sessions)}")
        for session in sessions[:3]:
            summary = session.get('summary', '')[:50]
            print(f" - {summary}...")

    finally:
        await memory.close()

    print(f"\n{'='*70}")
    print('Demo Complete!')
    print('Data persisted to CosmosDB for future sessions')
    print(f"{'='*70}")


await main()

Agent Memory Demo: Financial Advisor (CosmosDB)
Integration: Agent Framework ContextProvider
Memory: Automatic (no manual add_turn calls)
Backend: Azure CosmosDB (enterprise-grade)

CosmosDB: Endpoint configured

SESSION: Initial Consultation
[Orchestrator] Starting session 32684b07-99bc-4722-a305-ae29534579bd
[MemoryKeeper] Initializing session context for user: sarah_demo_cosmos
  ℹ No long-term insight found for user (will be created after sufficient sessions)
     📋 Session b304d194...: The session focused on optimizing Sarah’s tax strategy around retirement savings given her $150k inc...
     📋 Session 12ef1651...: The session focused on refining Sarah’s retirement investment strategy given her 30-year horizon and...
     📋 Session 0e85f9a4...: The session focused on Sarah’s retirement planning given her age (35), income ($150k), 30-year horiz...
     📋 Session 03e7bdec...: The user, a 35-year-old U.S.-based software engineer earning $150k and planning for retirement over ...
    